# Phase 3: Exploratory Data Analysis (EDA)
## Reliable Early Sepsis Prediction from Sparse and Non-Stationary ICU Time-Series
**Course:** M.Tech Data Science (24DS601 Machine Learning)  
**Investigator:** Akash J P (`CB.AI.P2DSC26033`), Amrita Vishwa Vidyapeetham  
**Dataset:** PhysioNet / Computing in Cardiology Challenge 2019  

---
### Notebook Objectives:
1. Ingest raw patient `.psv` files without data manipulation or temporal leakage.
2. Profile dataset dimensions, patient counts, and observation durations (`ICULOS`).
3. Conduct rigorous missing-value analysis across vital signs, laboratory tests, and demographics.
4. Quantify class imbalance at patient-level and hourly-row level.
5. Examine pre-imputation continuous feature distributions.
6. Visualize longitudinal trajectories of septic vs non-septic patients.
7. Inspect collinearity among physiological vital signs.

### Cell 1: Environment Setup, Package Imports & Reproducibility Policy

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is in sys.path
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from sepsis_ml.config import (
    ALL_COLUMNS, VITAL_SIGNS, LAB_VARIABLES, DEMOGRAPHIC_VARIABLES,
    TARGET_COL, RAW_DATA_DIR, FIGURES_DIR, TABLES_DIR
)
from sepsis_ml.reproducibility import seed_everything
from sepsis_ml.data_loader import iter_patient_files, load_patient_psv

seed = seed_everything(42)
print(f"[STATUS] Reproducibility seed locked to: {seed}")
print(f"[STATUS] Number of expected clinical covariates: {len(ALL_COLUMNS)-1}")

### Cell 2: Ingest Raw ICU Patient Cohort

In [2]:
# Load available patient records
patient_files = list(iter_patient_files(RAW_DATA_DIR))
print(f"Total patient files located in raw directory: {len(patient_files)}")

frames = [load_patient_psv(p) for p in patient_files]
raw_df = pd.concat(frames, ignore_index=True)

print(f"Unified Raw Cohort Shape: {raw_df.shape}")
print(f"Total Unique Patients: {raw_df['patient_id'].nunique()}")
raw_df.head()

### Cell 3: ICU Stay Duration and Monotonicity Audit

In [3]:
# Check ICULOS per patient
stay_summary = raw_df.groupby('patient_id')['ICULOS'].agg(['count', 'min', 'max'])
is_monotonic = raw_df.groupby('patient_id')['ICULOS'].apply(lambda s: s.is_monotonic_increasing).all()

print(f"All patient stay sequences monotonically increasing: {is_monotonic}")
print("ICU Stay Length Summary (Hours):")
print(stay_summary.describe())

### Cell 4: Class Imbalance Audit (Patient-Level vs Hourly-Level)

In [4]:
# Patient-level sepsis incidence
pt_sepsis = raw_df.groupby('patient_id')[TARGET_COL].max()
pt_counts = pt_sepsis.value_counts()
pt_prev = pt_sepsis.mean() * 100

# Row-level sepsis incidence
row_counts = raw_df[TARGET_COL].value_counts()
row_prev = raw_df[TARGET_COL].mean() * 100

print(f"Patient-level Sepsis Count: {dict(pt_counts)} -> Prevalence: {pt_prev:.2f}%")
print(f"Hourly-level Sepsis Count: {dict(row_counts)} -> Prevalence: {row_prev:.2f}%")
print(f"Imbalance Ratio (Hourly): 1 positive per {row_counts[0]/row_counts[1]:.1f} negative observations")

### Cell 5: Missingness and Measurement Density Stratification

In [5]:
# Compute missingness by feature and group
missing_rates = raw_df[ALL_COLUMNS[:-1]].isna().mean() * 100

vital_density = (100 - missing_rates[VITAL_SIGNS]).mean()
lab_density = (100 - missing_rates[LAB_VARIABLES]).mean()
demo_density = (100 - missing_rates[DEMOGRAPHIC_VARIABLES]).mean()

print(f"Average Vital Signs Sampling Density: {vital_density:.2f}%")
print(f"Average Laboratory Tests Sampling Density: {lab_density:.2f}%")
print(f"Average Demographic Reporting Density: {demo_density:.2f}%")

print("\nTop 10 Most Missing Clinical Variables:")
print(missing_rates.sort_values(ascending=False).head(10))

### Cell 6: Visualizing the 8 Core Diagnostic EDA Figures

In [6]:
# Display generated figures from outputs/figures
from IPython.display import Image, display

plots_to_show = [
    "01_missingness_percentage.png",
    "02_missingness_heatmap.png",
    "03_length_of_stay_distribution.png",
    "04_sepsis_class_distribution.png",
    "05_vital_sign_distributions.png",
    "06_patient_temporal_trajectories.png",
    "07_vital_correlation_matrix.png",
    "08_observation_frequency.png"
]

for p_name in plots_to_show:
    p_path = FIGURES_DIR / p_name
    if p_path.exists():
        print(f"=== Showing: {p_name} ===")
        display(Image(filename=str(p_path), width=700))
    else:
        print(f"Plot {p_name} not found on disk.")

### Cell 7: Correlation Analysis and Physiological Collinearity

In [7]:
# Compute correlation matrix for vital signs
corr = raw_df[VITAL_SIGNS].corr()
print("Vital Signs Pearson Correlation Matrix:")
display(corr.round(3))

# Strongest pairwise correlations
pairs = corr.unstack().sort_values(ascending=False)
pairs = pairs[pairs < 0.9999].drop_duplicates()
print("\nTop Pairwise Correlations in Vitals:")
print(pairs.head(5))